# Name
Please write your name down here:

Peter Brederlow

**Gene set enrichment analysis: pathways**

Having identified a number of differentially expressed genes between the CD and HFD diets on Day 5, we are interested in their functional composition. We will try to find gene classes that have an unusually large amount of differentially expressed genes (i.e. more than it would be expected by random chance).

Such gene classes can be defined in a number of ways: the chromosome they are found on, their function / properties (e.g. kinase activity, membrane protein), or the biochemical pathways they play a role in.

There are multiple data sources you can obtain such gene sets from. Position is found in the mouse genome annotation, function and properties in the Gene Ontology (GO) database, and biochemical pathways in KEGG, Reactome, etc. Today we will use KEGG pathways as our reference gene sets, but our analysis could easily be adapted to other kinds of gene sets.

In [1]:
# imports
import pandas as pd
import numpy as np
from Bio.KEGG.REST import *
from Bio.KEGG.KGML import KGML_parser
from Bio.Graphics.KGML_vis import KGMLCanvas
from IPython.display import Image
from io import StringIO
from scipy import stats
from statsmodels.stats.multitest import multipletests
from tqdm.notebook import tqdm

# KEGG REST service and visualizing KEGG pathways
You can find documentation about the [KEGG Rest interface](https://www.kegg.jp/kegg/docs/keggapi.html) itself and how to use it directly using URLs in a browser here: http://www.genome.jp/kegg/rest/keggapi.html

# KEGG and gene ID mapping
Familiarize yourself with the KEGG Rest interface and how to access it with Biopython.<br>
Before you start solving task 1.1, read its task description, look at and run all code examples given there. Try to understand what happens by recreating some REST calls as URL calls in your browser. The Biopython functions `kegg_list` and `kegg_link` simply wrap these URL calls into a function and return the response in plain text.

E.g. accessing the URL http://rest.kegg.jp/list/pathway/hsa lists all pathways associated with humans and is equivalent to calling `kegg_list('pathway', 'hsa').read()` in Python. (Organsim IDs: `hsa`: Homo Sapiens, `mmu`: Mus musculus)

# Extract gene lists for all (mouse) KEGG pathways and store them
Below is some example code showing how to get data out of the KEGG REST service in general.
It lists all the mouse pathways and extracts the pathway IDs from the REST response, then pulls the gene list for one pathway.

* The raw list of pathways looks like this:<br>
path:mmu00010	Glycolysis / Gluconeogenesis - Mus musculus (mouse)<br>
path:mmu00020	Citrate cycle (TCA cycle) - Mus musculus (mouse)<br>
path:mmu00030	Pentose phosphate pathway - Mus musculus (mouse)<br>
path:mmu00040	Pentose and glucuronate interconversions - Mus musculus (mouse)<br>
path:mmu00051	Fructose and mannose metabolism - Mus musculus (mouse)<br>
path:mmu00052	Galactose metabolism - Mus musculus (mouse)<br>
path:mmu00053	Ascorbate and aldarate metabolism - Mus musculus (mouse)<br>
...<br>
(path:ID)(TAB)(Description with spaces)


* The raw list of genes in a pathway looks like this:<br>
path:mmu00010	mmu:100042025<br>
path:mmu00010	mmu:103988<br>
path:mmu00010	mmu:106557<br>
path:mmu00010	mmu:110695<br>
path:mmu00010	mmu:11522<br>
path:mmu00010	mmu:11529<br>
path:mmu00010	mmu:11532<br>
...<br>
(path:ID)(TAB)(organismID:geneID)
<br>

The gene identifiers are so-called Entrez ID's, since KEGG chose to use the Entrez system.

* Extend the code to get lists of gene IDs for each pathway<br>
* Store the lists in a way that allows convenient lookup of all genes in each pathway.
A DataFrame called `pw_df` with three columns should do: `pathway_id`, `entrez_gene_id` and `pathway_desc` (description), where the pathway ID and description will be equal for all genes from one pathway. From this you can extract all genes of a pathway using the first column. Be aware that neither `pathway_id` nor `entrez_gene_id` are unique identifiers, as each gene may be present in multiple pathways: it's a typical case of N-to-N ("many-to-many") mapping.
 
Beware: You can make a single call and retreive all pathways for mice in one go.

In [2]:
 # get all pathways from organism 'mmu' (Mus musculus)
pathways_response = kegg_list('pathway', 'mmu').read()
#Format = path:ID Description

# example for one pathway:
#example_gene_list = kegg_link('mmu', 'path:mmu00010').read()
# print(exampleGeneList)

# A neat trick to turn this directly into a pandas DataFrame
# is to use StringIO to simulate a CSV input file:

# Split the response on "\n" newline characters, split the lines on "\t" characters.
# Or read on a bit and adapt the trick in the next code block.
# Either way, store them in a pandas Series indexed by pathway ID's.

# use kegg_link(organism, pathway).read() to get the list of genes for each pathway

pw_ids = pd.Series(pathways_response.strip().split("\n"))
# YOUR CODE HERE
pw_df = pd.DataFrame(pw_ids.str.split("\t").tolist(), columns=["pathway_id", "description"])


In [3]:
gene_list = kegg_link('mmu', 'pathway').read()
gene_list

'path:mmu00010\tmmu:103988\npath:mmu00010\tmmu:106557\npath:mmu00010\tmmu:110695\npath:mmu00010\tmmu:11522\npath:mmu00010\tmmu:11529\npath:mmu00010\tmmu:11532\npath:mmu00010\tmmu:115487111\npath:mmu00010\tmmu:11669\npath:mmu00010\tmmu:11670\npath:mmu00010\tmmu:11671\npath:mmu00010\tmmu:11674\npath:mmu00010\tmmu:11676\npath:mmu00010\tmmu:12183\npath:mmu00010\tmmu:13382\npath:mmu00010\tmmu:13806\npath:mmu00010\tmmu:13807\npath:mmu00010\tmmu:13808\npath:mmu00010\tmmu:14120\npath:mmu00010\tmmu:14121\npath:mmu00010\tmmu:14377\npath:mmu00010\tmmu:14378\npath:mmu00010\tmmu:14433\npath:mmu00010\tmmu:14447\npath:mmu00010\tmmu:14751\npath:mmu00010\tmmu:15275\npath:mmu00010\tmmu:15277\npath:mmu00010\tmmu:16828\npath:mmu00010\tmmu:16832\npath:mmu00010\tmmu:16833\npath:mmu00010\tmmu:17330\npath:mmu00010\tmmu:18534\npath:mmu00010\tmmu:18597\npath:mmu00010\tmmu:18598\npath:mmu00010\tmmu:18641\npath:mmu00010\tmmu:18642\npath:mmu00010\tmmu:18648\npath:mmu00010\tmmu:18655\npath:mmu00010\tmmu:18663\npath

In [4]:
gene_list = gene_list.strip().split("\n")
gene_list = [line.split("\t") for line in gene_list]
gene_list_df = pd.DataFrame(gene_list, columns=["pathway_id", "entrez_id"])
gene_list_df['pathway_id'] = gene_list_df['pathway_id'].str.replace("path:mmu", "mmu")

In [5]:
assert 'pw_df' in locals()

## Create the pathway-to-gene DataFrame and store it in a CSV
**How many rows does the `pw_entrez` DataFrame have? How many unique Entrez identifiers are in there?**
Store the DataFrame as a csv file so that you can load it back up easily if necessary.

In [6]:
pw_entrez = pd.merge(gene_list_df, pw_df, on="pathway_id", how="left")

# Convert Gene Entrez ID to Gene Identifier Format

http://www.informatics.jax.org/downloads/reports/MGI_Gene_Model_Coord.rpt <br>
The file above contains mappings between different identifier types, including the Entrez IDs that KEGG uses and the gene symbols we have in the DE data.
Download and read in this file, then use the respective columns to map the Entrez ID numbers to gene symbols.

## Create a `mapping_df` DataFrame with two columns:
* `gene` (to match the index name we had used in the DE notebook) and
* `entrez_gene_id` to match our KEGG `pathway_entrez` table.

<div class="alert alert-block alert-warning"><b>Watch out! </b> Pandas may convert the Entrez ID's to floating point numbers when loading the csv. Also, the "mmu:" prefix which KEGG uses is missing from them. Turn those <code>12345.0</code>-like floating point values to <code>mmu:12345</code> strings, otherwise you will have a hard time matching them with KEGG pathways.</div>

In [7]:
# YOUR CODE HERE
url = 'https://raw.githubusercontent.com/Practical-Integrative-Bioinformatics/Introduction/refs/heads/main/data/MGI_Gene_Model_Coord.rpt'
mapping_df = pd.read_csv(url, sep='\t', index_col=False, dtype=str).dropna(subset=['6. Entrez gene id'])
mapping_df = mapping_df.iloc[:,[2, 5]]
mapping_df.columns = ['gene_symbol', 'entrez_id']
mapping_df['entrez_id'] = 'mmu:' + mapping_df['entrez_id']
mapping_df

,gene_symbol,entrez_id
0,a,mmu:50518
1,Pzp2,mmu:11287
2,Abl1,mmu:11350
3,Abl2,mmu:11352
4,Scgb1b27,mmu:11354
...,...,...
106913,Rr406208,mmu:132445428
106914,Rr406209,mmu:132445429
106915,Rr562,mmu:171544
106916,Rr563,mmu:171545


In [8]:
assert 'mapping_df' in locals()

## Merge `pw_df` with your Mapping DataFrame

Your `pw_df2` should contain only these four columns:
1. `pathway_id`
2. `pathway_desc`
3. `entrez_gene_id`
4. `gene`

In [9]:
# YOUR CODE HERE
pw_df2 = pd.merge(pw_entrez, mapping_df, on="entrez_id", how="left")
pw_df2.columns = ['pathway_id', 'entrez__gene_id', 'pathway_desc', 'gene']
pw_df2

,pathway_id,entrez__gene_id,pathway_desc,gene
0,mmu00010,mmu:103988,Glycolysis / Gluconeogenesis - Mus musculus (h...,Gck
1,mmu00010,mmu:106557,Glycolysis / Gluconeogenesis - Mus musculus (h...,Ldhal6b
2,mmu00010,mmu:110695,Glycolysis / Gluconeogenesis - Mus musculus (h...,Aldh7a1
3,mmu00010,mmu:11522,Glycolysis / Gluconeogenesis - Mus musculus (h...,Adh1
4,mmu00010,mmu:11529,Glycolysis / Gluconeogenesis - Mus musculus (h...,Adh7
...,...,...,...,...
41159,mmu05418,mmu:723893,Fluid shear stress and atherosclerosis - Mus m...,Mir10a
41160,mmu05418,mmu:74769,Fluid shear stress and atherosclerosis - Mus m...,Pik3cb
41161,mmu05418,mmu:75600,Fluid shear stress and atherosclerosis - Mus m...,Calml4
41162,mmu05418,mmu:75886,Fluid shear stress and atherosclerosis - Mus m...,Gstt4


In [10]:
assert 'pw_df2' in locals()

# Gene Set Enrichment

## Prepare your differential expression data

* Read in the `diffexpr` csv from `https://github.com/Practical-Integrative-Bioinformatics/Introduction/blob/main/data/diffexpr.csv?raw=True'`.
* Ensure you have the boolean column `is_de`, with criteria that you can perhaps adjust later. You can start by using $\text{p\_corr} < 0.01$ and $|\text{log2fold}| > 0.2$.

In [11]:
# YOUR CODE HERE
url = 'https://github.com/Practical-Integrative-Bioinformatics/Introduction/blob/main/data/diffexpr.csv?raw=True' 
diffexpr = pd.read_csv(url, index_col=0)
#diffexpr = diffexpr.drop_duplicates(subset='gene')
#diffexpr.index = diffexpr['gene']

In [12]:
assert 'diffexpr' in locals()

## Merge `diffexpr` with `pw_df2`

Create a new DataFrame `pw_de` that contains the information of both DataFrames.

In [20]:
# YOUR CODE HERE
pw_de = pd.merge(pw_df2, diffexpr, left_on='gene', right_index=True, how='left')
pw_de

,pathway_id,entrez__gene_id,pathway_desc,gene,log2fold,p_mwu,p_wilcoxon,p_student,p_corr,interesting
0,mmu00010,mmu:103988,Glycolysis / Gluconeogenesis - Mus musculus (h...,Gck,-0.506026,0.000006,0.000087,0.000014,0.000042,True
1,mmu00010,mmu:106557,Glycolysis / Gluconeogenesis - Mus musculus (h...,Ldhal6b,0.051538,0.132568,0.171134,0.224859,0.205337,False
2,mmu00010,mmu:110695,Glycolysis / Gluconeogenesis - Mus musculus (h...,Aldh7a1,-0.013718,0.760515,0.557162,0.630645,0.821831,False
3,mmu00010,mmu:11522,Glycolysis / Gluconeogenesis - Mus musculus (h...,Adh1,-0.030051,0.228485,0.058408,0.179858,0.319900,False
4,mmu00010,mmu:11529,Glycolysis / Gluconeogenesis - Mus musculus (h...,Adh7,-0.035641,0.199067,0.243011,0.334856,0.286160,False
...,...,...,...,...,...,...,...,...,...,...
41159,mmu05418,mmu:723893,Fluid shear stress and atherosclerosis - Mus m...,Mir10a,NaN,NaN,NaN,NaN,NaN,NaN
41160,mmu05418,mmu:74769,Fluid shear stress and atherosclerosis - Mus m...,Pik3cb,-0.037821,0.220872,0.090139,0.198644,0.311213,False
41161,mmu05418,mmu:75600,Fluid shear stress and atherosclerosis - Mus m...,Calml4,-0.201256,0.000515,0.000068,0.000948,0.001851,True
41162,mmu05418,mmu:75886,Fluid shear stress and atherosclerosis - Mus m...,Gstt4,0.177385,0.000013,0.000007,0.000007,0.000080,False


In [14]:
assert 'pw_de' in locals()

## Perform gene set enrichment with the KEGG gene sets you extracted
Several tests are suitable for our purpose: the $\chi^2$ test we had used before, Fisher's exact test, or the simplest binomial test.<br>

Let's use the binomial test today. There is a relatively small fraction of genes in any one pathway, so the "not in pathway" row of the contingency tables would be nearly constant, and much larger than the "in pathway" row. So a binomial simplification is perfectly suited to our case

To do a binomial test, you will first have to determine the "number of trials" / "number of successes" / "probability of success" parameters. Remember, the latter is independent of your pathway, so you'll only have to calculate it once, and use the same parameter for all pathways. (This is why the binomial test is a simplification.)

Avoid writing `for` loops here. Use pandas' efficient grouping and aggregation methods.

Your `pw_summary` DataFrame should have the following columns:
* `pathway_id` as index
* `num_genes` number of genes in pathway
* `num_de` number of interesting differentially expressed genes in pathway
* `p_binom` p-values of enrichment (binomial test)
* `p_corr` multiple testing corrected binomial test p values
* `p_desc` description of pathway

**Details to think about: How do you want to treat Entrez ID's that have no corresponding measurements from the microarray experiment? What about those that have multiple?**

In [21]:
p_success = (diffexpr['interesting'] == True).sum() / len(diffexpr)
p_success

np.float64(0.13931760934279827)

In [26]:
# YOUR CODE HERE
pw_de = pw_de.dropna(subset=['log2fold']) #drop pathway-gene pairs where the gene is not in the diffexpr dataset
pw_summary = pw_de['pathway_id'].drop_duplicates().to_frame()
pw_summary['description'] = pw_summary['pathway_id'].apply(lambda x: pw_de[pw_de['pathway_id'] == x]['pathway_desc'].iloc[0])
pw_summary['num_genes'] = pw_summary['pathway_id'].apply(lambda x: pw_de[pw_de['pathway_id'] == x].count()['gene'])
pw_summary['num_de'] = pw_summary['pathway_id'].apply(lambda x: pw_de[(pw_de['pathway_id'] == x) & (pw_de['interesting'] == True)]['gene'].count())
pw_summary['p_binom'] = pw_summary.apply(lambda row: stats.binomtest(row['num_de'], row['num_genes'], p_success).pvalue, axis=1)
pw_summary

,pathway_id,description,num_genes,num_de,p_binom
0,mmu00010,Glycolysis / Gluconeogenesis - Mus musculus (h...,60,13,0.092179
67,mmu00020,Citrate cycle (TCA cycle) - Mus musculus (hous...,31,2,0.304641
99,mmu00030,Pentose phosphate pathway - Mus musculus (hous...,32,5,0.797024
133,mmu00040,Pentose and glucuronate interconversions - Mus...,24,8,0.013014
169,mmu00051,Fructose and mannose metabolism - Mus musculus...,32,6,0.440073
...,...,...,...,...,...
40387,mmu05414,Dilated cardiomyopathy - Mus musculus (house m...,102,10,0.255349
40497,mmu05415,Diabetic cardiomyopathy - Mus musculus (house ...,171,10,0.001233
40706,mmu05416,Viral myocarditis - Mus musculus (house mouse),83,15,0.267503
40798,mmu05417,Lipid and atherosclerosis - Mus musculus (hous...,207,31,0.687668


I excluded multiple rows and rows without a measurement, the latter since they would influence the succes probability without having any "option to succeed", the former since it would be overrepresented in num_de and hence skew the p-value.

## Extract a list of significantly enriched KEGG pathways
**How many pathways are enriched? How many survive the Benjamini-Hochberg correction? Do you think your criteria for `interesting` differentially expressed genes were suitable?**

In [27]:
# YOUR CODE HERE
pw_summary['p_adj'] = multipletests(pw_summary['p_binom'], method='fdr_bh')[1]
pw_sig = pw_summary[pw_summary['p_adj'] < 0.05]
pw_sig

,pathway_id,description,num_genes,num_de,p_binom,p_adj
426,mmu00140,Steroid hormone biosynthesis - Mus musculus (h...,82,29,9.234971e-07,0.000267
2816,mmu00830,Retinol metabolism - Mus musculus (house mouse),83,25,1.668974e-04,0.008703
7235,mmu03040,Spliceosome - Mus musculus (house mouse),117,5,1.164082e-03,0.032138
8383,mmu03082,ATP-dependent chromatin remodeling - Mus muscu...,85,2,4.816381e-04,0.017580
13081,mmu04120,Ubiquitin mediated proteolysis - Mus musculus ...,130,2,1.463477e-06,0.000267
13412,mmu04140,Autophagy - animal - Mus musculus (house mouse),151,6,8.651025e-05,0.006315
21787,mmu04714,Thermogenesis - Mus musculus (house mouse),183,9,1.534537e-04,0.008703
27987,mmu04940,Type I diabetes mellitus - Mus musculus (house...,57,17,1.677701e-03,0.038273
29424,mmu05012,Parkinson disease - Mus musculus (house mouse),224,12,6.323081e-05,0.005770
29689,mmu05014,Amyotrophic lateral sclerosis - Mus musculus (...,298,22,5.715072e-04,0.018964


Almost all of the pathways survive multiple testing correction. This is highly unusual.

# KEGG map visualization

Reminder: http://nbviewer.jupyter.org/github/widdowquinn/notebooks/blob/master/Biopython_KGML_intro.ipynb

For Python (in addition to the Biopyhton module) https://github.com/idekerlab/py2cytoscape in combination with https://github.com/idekerlab/KEGGscape may be another alternative.

Generally speaking, it's a good idea to make use of multiple pathway databases like Reactome or WikiPathways, but in this course we will restrict ourselves to visualizing native KEGG pathways using Biopython.

## Pick some significantly enriched KEGG pathways of your choice and visualize them
The simplest way is to get the images directly using `kegg_get` and draw them using `Image(kegg_get(...))`. <br>
This will literally just pull an PNG image file from the KEGG website and display it.

In [18]:
# YOUR CODE HERE
raise NotImplementedError()

NotImplementedError: 

## Define a color scheme respresenting whether a gene is significantly differentially expressed or not

To have more control over the visualization we will extract the pathway in "KGML" (KEGG XML) format. This will allow us to change attributes of the graph before drawing it.
We will color the nodes by differential expression.

Use three colors:
1. not significant: <span style='color:#BFFFBF'>in green</span>
2. significantly overexpressed in CD: <span style='color:#FF0000'>in red</span>
3. significantly overexpressed in HFD: <span style='color:#0000FF'>in blue</span>

If you have enough time, you can use a continuous color gradient from color 2 to 3 based on the continuous p-value and/or log2 fold change values. For now we will just save the color codes in variables to be able to easily change them later.

In [ ]:
not_significant='#BFFFBF'
CD_significant='#FF0000' # red
HFD_significant='#0000FF' # blue

## Visualize the pathway from before with color-coded enzyme nodes

You may need to define a suitable mapping from single genes to what is actually shown in the pathway map. The main problem you will encounter is that there can be multiple genes perfoming an enzymatic function represented by a single box. In this case, you will have to make a decision how to display potentially conflicting information (i.e. how to color the node). Also note that not all genes have been necessarily measured by the microarray.

Use `KGMLCanvas(pathway, import_imagemap=True).draw(PDF_filename)` to draw them and save them as a PDF file.

In [ ]:
def visualize_pathway(pathway_name):

    pathway = KGML_parser.read(kegg_get(pathway_name, "kgml"))
    print('Pathway description:')
    print(pathway)
    # print('Loop over genes:')
    for gene in pathway.genes:  # already confusing: this should be called enzyme node...
        # print(gene.name)
        
        gene_ids = gene.name.split()
        gene_df = de_pw.loc[de_pw['entrez_gene_id'].isin(gene_ids)]
        
        if gene_df.shape[0] == 0:
            continue
        
        # print('New gene:')    
        # print(gene_ids)
        # Notice how one print of "gene.name" can sometimes give you a space-separated string of multiple gene names.
        # Transform this string into a list of names, and look up their fold changes and significance (is_de).
        # Reduce it to a single category (non-DE, overexpressed in CD, overexpressed in HFD) based on criteria of your choice.
        
        # If no genes are available for a node, skip to the next iteration.
        
        is_de = gene_df['is_de'].mean() > 0.3
        higher_in_hfd = gene_df['log2fold'].mean() < 0                
        
        if True: # if this pathway node is contained in our DE data...
            if is_de: # if any of the gene names corresponds to any significantly expressed gene... (and put the actual fold-change into a variable)
                for graphic in gene.graphics: # there might be multiple graphics objects for a node, change all their colors
                    # This is pretty bad design from KGMLCanvas: boxes will be plotted over each other in this case,
                    # but we'll deal with it by always plotting the same color
                    # print('Current gene color:')
                    # print(graphic.bgcolor) # this is the standard color you need to change
                    # we know the gene is significantly differentially expressed, but what direction?
                    if higher_in_hfd:  # if CD overexpressed
                        graphic.bgcolor = HFD_significant
                    else: # if HFD overexpressed
                        graphic.bgcolor = CD_significant
            else: # not differentially expressed, but contained in our gene list
                for graphic in gene.graphics:
                    graphic.bgcolor = not_significant

    # these lines will save a PDF file to pdf_file_path
    pdf_file_path = f'{pathway_name.replace(":", "_")}.pdf'
    KGMLCanvas(pathway, import_imagemap=True).draw(pdf_file_path)

visualize_pathway('path:mmu00140')

## Pathway intersections

Many genes can be found in multiple pathways. Look for differentially expressed genes in an enriched pathway that are part of at least one other pathway. Visualize two such pathways using your color scheme from task 3. **How many other differentially expressed genes are in this second pathway? Are there similarities between these two pathways, e.g. are the differentially expressed genes common to both pathways in a similar context?**

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

# Optional: Advanced KEGG map functions
## Change other attributes of the map

Besides the background color, you have access to other attributes of the nodes, such as labels. You can try setting them to gene symbols, keeping in mind the overlapping

The labels on the nodes can be very confusing, containing a long list of alternative gene symbols for each gene.

Take a pathway with multiple labels per node and change the labels of each node to the gene name from our DE data.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()